# Canonical Model 00 · Conceptual Model and Build

Every notebook in this set uses **one** model: an irregular DISV/Voronoi
**alluvial valley**. Two tributary streams enter the mountain front, **converge**
at a confluence, and the combined main stem discharges to a single terminal
**lake** near the valley mouth. The streams **gain** from the aquifer in the
headwaters and **lose** to it downgradient, where a cross-valley bedrock
constriction steps the water table down and the lake perches above it.

> **Purpose:** establish one trusted, package-rich model whose physics,
> observations, visual diagnostics, particle tracking, parallel splitting, and
> PEST calibration can all be exercised against the same truth.

### Four hydrostratigraphic layers

| Layer | Unit | Hydrogeologic role |
|---|---|---|
| 1 | Upper unconfined alluvium | Water table, UZF infiltration, stream & spring **seepage** |
| 2 | Lower unconfined (main) aquifer | Regional flow, shallow pumping, high-K paleochannel |
| 3 | Lacustrine clay aquitard | Distinct low-K; damps vertical communication |
| 4 | Confined basin-fill aquifer | Deep pumping cone, confined response |

### Boundary conditions
CHD (up-valley inflow) · GHB (valley-mouth outflow) · RCH (areal recharge) ·
UZF (unsaturated-zone recharge incl. an **infiltration** pond) · WEL (shallow +
deep pumping) · DRN (two aquifer **seepage** spring slopes) · SFR (the converging
stream network) · LAK (the terminal lake) · MVR (main stem → lake).

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
import myflopy as mf
from canonical_notebook_style import notebook_header

notebook_header('00', 'Conceptual Model and Build', 'One valley model, one contract, every workflow.')

workspace = Path('../artifacts/canonical_master')
config = mf.CanonicalModelConfig.validation()   # 50x50; use CanonicalModelConfig() for the full >=10,000-cell profile
model = mf.build_canonical_model(workspace / 'gwf', config=config, name='canonical_master')

# The contract checks grid type, four-layer hydrostratigraphy, the full package
# set, the named observation targets, and that surface-water cells are refined.
mf.CANONICAL_MODEL_CONTRACT.validate(model)
model.regions.region_summary()

## What the build contains

The `region_summary` above lists the named feature regions the rest of the set
queries by name: `north_seepage_springs` / `south_seepage_springs` (DRN slopes),
`infiltration_pond` (UZF), `all_streams` (the converging SFR network),
`all_lakes` / `lake_zone_valley_lake` (the terminal lake), and `uzf_active`
(the valley floor). Surface-water cells are deliberately **refined** so the
stream corridor and lake resolve on finer Voronoi cells.

In [ ]:
success, report = model.run_simulation()
assert success, '\n'.join(report[-30:])

# Head signals: a strong regional gradient in every layer, transient movement in
# the unconfined aquifer, and a distinct (smaller) confined response.
mf.canonical_head_signals(model)

## Mass balance: the first thing to check

Before reading any map, confirm the model **converged and conserves water**.
The MF6 GWF listing budget gives the volumetric in/out by component and the
percent discrepancy. **What to look for:** a percent discrepancy near zero
(well under 1%), and a sensible balance — up-valley CHD inflow and areal/UZF
recharge in; GHB outflow at the mouth, stream and drain seepage, and pumping out.

In [ ]:
import flopy
import pandas as pd

gwf_lst = next(p for p in Path(model.gwf.model_ws).glob('*.lst') if p.name != 'mfsim.lst')
incremental, _cumulative = flopy.utils.Mf6ListBudget(str(gwf_lst)).get_dataframes()
final = incremental.iloc[-1]

inflows = final[[c for c in final.index if c.endswith('_IN') and final[c] != 0]].sort_values(ascending=False)
outflows = final[[c for c in final.index if c.endswith('_OUT') and final[c] != 0]].sort_values(ascending=False)
budget = pd.concat([inflows.rename('volume'), outflows.rename('volume')]).to_frame()
print(f"percent discrepancy (final step): {final['PERCENT_DISCREPANCY']:.4f} %")
assert abs(final['PERCENT_DISCREPANCY']) < 1.0
budget

## Is it physically sensible?

Two signatures define this valley and recur throughout the set:
1. The stream is mostly **gaining** in the headwaters and turns **losing** toward
   the lake — so `gaining_reach_count` should dominate but `losing_reach_count`
   is nonzero.
2. The terminal lake **perches**: its stage sits above its bottom and above the
   downgradient water table, so it leaks downward to the aquifer (shown in 02).

In [ ]:
import pandas as pd

sfr = mf.canonical_sfr_signals(model)
lake_stage = model.targets.lake_stage.simulated_series()['valley_lake']
pd.Series({
    'reaches': sfr['reach_count'],
    'gaining_reaches': sfr['gaining_reach_count'],
    'losing_reaches': sfr['losing_reach_count'],
    'min_stream_depth_ft': round(sfr['minimum_depth'], 3),
    'lake_stage_ft': round(float(lake_stage.iloc[-1]), 2),
    'lake_bottom_ft': 96.0,
}, name='valley signals')

## Next

The model builds, runs, conserves water, and behaves like a real valley.
Continue to **01 · Packages and Observations** to see how each physical feature
becomes a named, queryable observation that both the plots and PEST consume.